In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Daily hydrometeorological preprocessing

This notebook preprocesses raw time series of daily catchment-aggregated hydrometeorological data (runoff, temp, precip) for later machine learning applications.

For each catchment:

- Standardise variable names
- Ensure all expected variables exist (missing variables are added as `NaN`)
- Linearly interpolate runoff and discharge gaps up to 3 days and flag interpolated runoff values
- Calculate:
  - log runoff (`log(Q + 0.02)`)
  - runoff normalised by site-specific Q95
  - runoff ECDF (site-specific runoff percentile)
- Generate antecedent-condition features, summing precip and runoff and averaging temperature over n_windows prior to and including the day of observation. n_windows is a list, e.g. 7, 30, 90 for three aggregation periods.
- Compute annual runoff, precipitation, and temperature summaries
- Compute long-term mean annual climatologies for each catchment

**Outputs:**

- **Daily data**: Processed daily data are written as **one Parquet file per catchment**. Why Parquet? Much smaller than CSV, faster to read/write, preserves data types, allows reading only selected columns later, scales better for large ML workflows.

    Note that the original approach of concatenating all catchments into a single dataframe exceeded available memory (~140 million daily records). Writing site-level Parquet files avoids this bottleneck while still allowing efficient construction of ML training tables later.

- **Annual summaries**: `gts_q-met_annual.csv`. One row per site-year containing annual runoff, precipitation, and mean temperature.

- **Catchment averages (static)**: `gts_q-met_annual_mean.csv`. One row per catchment containing Q95, data-availability flags, and long-term mean annual climatologies.

In [2]:
# Paths to folders

parent_dir = Path(r"/home/jovyan/shared/common/norsink/vm_catchments_data/gridded_by_site")

# raw_dir = os.path.join(parent_dir, "raw")
raw_dir = parent_dir / "raw"  # instead of os.path.join, this works if have imported Path
daily_out_dir = parent_dir / "processed" / "daily"
annual_out_dir = parent_dir / "processed" / "annual"
static_out_dir = parent_dir / "processed" / "static"

for d in [daily_out_dir, annual_out_dir, static_out_dir]:
    d.mkdir(parents=True, exist_ok=True)

In [3]:
# Get list of raw csv files & check one file
raw_csv_files = sorted(raw_dir.glob("*.csv"))

print(f"{len(raw_csv_files):,} files found")

df = pd.read_csv(raw_csv_files[0])

print(df.columns)
df.head()

8,482 files found
Index(['gts_avg_gwb_q_mm', 'gts_avg_rr_mm', 'gts_avg_tm_c', 'date',
       'gts_mod_discharge_m3ps', 'site_code'],
      dtype='object')


,gts_avg_gwb_q_mm,gts_avg_rr_mm,gts_avg_tm_c,date,gts_mod_discharge_m3ps,site_code
0,0.7553,4.0095,-4.4100,1980-01-01,8.8160,001-119928
1,0.7030,0.0000,-7.6395,1980-01-02,8.2060,001-119928
2,0.6603,0.0015,-12.5070,1980-01-03,7.7074,001-119928
3,0.6201,0.0015,-18.5565,1980-01-04,7.2381,001-119928
4,0.5950,0.7000,-18.4190,1980-01-05,6.9449,001-119928


# Functions

In [4]:
def add_rolling_feature(df, col, out_prefix, window, stat):

    if stat == "sum":
        df[f"{out_prefix}_{window}d_sum"] = (
            df[col]
            .rolling(window=window, min_periods=window)
            .sum()
        )

    elif stat == "mean":
        df[f"{out_prefix}_{window}d_mean"] = (
            df[col]
            .rolling(window=window, min_periods=window)
            .mean()
        )

    return df

# Process all sites

In [ ]:
raw_csv_files = sorted(raw_dir.glob("*.csv"))
# test_files = raw_csv_files[:10]

# Define rolling aggregation features (col, statistic, has_data)
rolling_windows = [7, 30, 90]
# N.B. this is currently aggregated for all features (runoff, precip, temp)

rename_col_dict = {
    'gts_avg_gwb_q_mm': 'gts_runoff_daily',
    'gts_avg_rr_mm': 'gts_precip_daily',
    'gts_avg_tm_c': 'gts_temp_daily',
    'gts_mod_discharge_m3ps': 'gts_q_daily_m3ps'
}

interp_gaps_limit = 3  # days
epsilon = 0.02

# From notebook 03a
st_dt = "1980-01-01"
end_dt = "2025-12-31"

# ----- Start processing ----------

required_cols = list(rename_col_dict.values())

site_summary_list = []
annual_list = []
# daily_list = []  # Not if have too many catchs, then save site-by-site

full_dates = pd.date_range(
    start=st_dt,
    end=end_dt,
    freq="D",
)

for file in tqdm(raw_csv_files):

    # Read
    df = pd.read_csv(file)

    # Rename columns
    df = df.rename(columns=rename_col_dict)

    # Ensure all expected columns exist, if not add them with nans for consistency
    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Date (and sort, to guarantee a clean index before rolling calcs)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    site_code = str(df["site_code"].iloc[0])

    # Check for duplicate dates
    if df["date"].duplicated().any():
        raise ValueError(f"{site_code}: duplicate dates detected")
   
    # Check for complete daily sequence
    if len(df) != len(full_dates):
        missing_dates = full_dates.difference(df["date"])
    
        raise ValueError(
            f"{file.name}: {len(missing_dates)} missing dates "
            f"between {full_dates.min().date()} and "
            f"{full_dates.max().date()}"
        )

    # -------------------------
    # Data availability (within-col)
    has_runoff = df["gts_runoff_daily"].notna().any()
    has_precip = df["gts_precip_daily"].notna().any()
    has_temp = df["gts_temp_daily"].notna().any()
    has_q = df["gts_q_daily_m3ps"].notna().any()

    # -------------------------
    # Runoff processing
    if has_runoff:

        # ------------------------ 
        # Interpolate small gaps in runoff, if there are data to interpolate

        # Save missing-value flags before interpolation
        runoff_missing_before = df["gts_runoff_daily"].isna()

        df["gts_runoff_daily"] = df["gts_runoff_daily"].interpolate(
            method="linear",
            limit=interp_gaps_limit,
            limit_area="inside",
        )
        df["gts_q_daily_m3ps"] = df["gts_q_daily_m3ps"].interpolate(
            method="linear",
            limit=interp_gaps_limit,
            limit_area="inside",
        )

        # Flag
        df["flag_runoff_interp"] = (
            runoff_missing_before &
            df["gts_runoff_daily"].notna()
        )

        # ------------------------
        # Calculate additional derived runoff variables
        q95 = df["gts_runoff_daily"].quantile(0.95)

        # Log runoff. Add small value (epsilon) to avoid log0
        df["gts_runoff_daily_log"] = np.log(df["gts_runoff_daily"] + epsilon)
        # N.B. this can be invalid if runoff < -epsilon. Check
        if (df["gts_runoff_daily"] < 0).any():
            print(site_code, "has negative runoff values!")

        # Relative runoff - normalised by q95, i.e. how extreme is the Q?
        # Shouldn't be 0, but just in case
        if q95 > 0:
            df["gts_runoff_daily_rel_q95"] = df["gts_runoff_daily"] / q95
        else:
            df["gts_runoff_daily_rel_q95"] = np.nan

        # Relative runoff, the ECDF (or percentile corresponding to daily runoff)
        df["gts_runoff_daily_ecdf"] = (df["gts_runoff_daily"].rank(method="average", pct=True))

    else:
        q95 = np.nan

        df["flag_runoff_interp"] = False
        df["gts_runoff_daily_log"] = np.nan
        df["gts_runoff_daily_rel_q95"] = np.nan
        df["gts_runoff_daily_ecdf"] = np.nan

    # -----------------------
    rolling_features = [
        ("gts_runoff_daily", "gts_runoff", "sum", has_runoff),
        ("gts_precip_daily", "gts_precip", "sum", has_precip),
        ("gts_temp_daily", "gts_temp", "mean", has_temp),
    ]

    # Calculate rolling features (antecedent conditions)
    for col, out_prefix, stat, has_data in rolling_features:
    
        suffix = "sum" if stat == "sum" else "mean"
    
        if has_data:
            for window in rolling_windows:
                add_rolling_feature(
                    df,
                    col=col,
                    out_prefix=out_prefix,
                    window=window,
                    stat=stat,
                )
    
        else:
            for window in rolling_windows:
                df[f"{out_prefix}_{window}d_{suffix}"] = np.nan

    # ------------------------
    # Annual summaries
    df["year"] = df["date"].dt.year

    annual = (
        df.groupby("year")
          .agg(
              runoff_annual=(
                  "gts_runoff_daily",
                  lambda x: x.sum(min_count=1)
              ),
              precip_annual=(
                  "gts_precip_daily",
                  lambda x: x.sum(min_count=1)
              ),
              temp_annual=("gts_temp_daily", "mean"),
          )
          .reset_index()
    )

    annual["site_code"] = site_code

    annual_list.append(annual)

    # ------------------------
    # Long-term means
    site_summary_list.append({
        "site_code": site_code,
        "gts_runoff_q95": q95,
        "has_runoff": has_runoff,
        "has_precip": has_precip,
        "has_temp": has_temp,
        "runoff_mean_annual": annual["runoff_annual"].mean(),
        "precip_mean_annual": annual["precip_annual"].mean(),
        "temp_mean_annual": annual["temp_annual"].mean(),
    })

    # ---------------------
    # Daily table formatting
    df = df.drop(columns="year")

    # Move site_code and date to front of df
    front_cols = ["site_code", "date"]
    df = df[
        front_cols +
        [c for c in df.columns if c not in front_cols]
    ]

    # ----------------------
    # Write daily parquet per site
    # daily_list.append(df)  # use this if not many sites
    df.to_parquet(
        daily_out_dir / f"{site_code}.parquet",
        index=False,
        compression="zstd",
    )

# -------- Save annual and static ---------------

# Annual (with rounding)
annual_df = pd.concat(
    annual_list,
    ignore_index=True
).round(2)

annual_df.to_csv(
    annual_out_dir / "gts_q-met_annual.csv",
    index=False
)

# Static (mean of annual values)
site_summary_df = pd.DataFrame(site_summary_list).round(3)

site_summary_df.to_csv(
    static_out_dir / "gts_q-met_annual_mean.csv",
    index=False
)

  0%|          | 0/8482 [00:00<?, ?it/s]

In [15]:
# Look at last daily df
df

,site_code,date,gts_runoff_daily,gts_precip_daily,gts_temp_daily,gts_q_daily_m3ps,flag_runoff_interp,gts_runoff_daily_log,gts_runoff_daily_rel_q95,gts_runoff_daily_ecdf,gts_runoff_7d_sum,gts_runoff_30d_sum,gts_runoff_90d_sum,gts_precip_7d_sum,gts_precip_30d_sum,gts_precip_90d_sum,gts_temp_7d_mean,gts_temp_30d_mean,gts_temp_90d_mean
0,314-89994,1980-01-01,1.00,4.20,-4.75,0.0356,False,0.019803,0.289855,0.642929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,314-89994,1980-01-02,0.90,0.00,-7.95,0.0320,False,-0.083382,0.260870,0.602666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,314-89994,1980-01-03,0.90,0.00,-13.00,0.0320,False,-0.083382,0.260870,0.602666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,314-89994,1980-01-04,0.85,0.00,-18.80,0.0303,False,-0.139262,0.246377,0.579871,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,314-89994,1980-01-05,0.85,0.20,-19.70,0.0303,False,-0.139262,0.246377,0.579871,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16797,314-89994,2025-12-27,1.90,0.00,-3.70,0.0677,False,0.652325,0.550725,0.843917,18.15,91.55,189.40,0.50,88.80,295.85,-3.592857,2.096667,3.498889
16798,314-89994,2025-12-28,1.70,0.25,-0.05,0.0605,False,0.542324,0.492754,0.813832,16.05,91.10,190.15,0.55,80.35,296.10,-3.685714,1.946667,3.387778
16799,314-89994,2025-12-29,1.55,0.10,-4.95,0.0552,False,0.451076,0.449275,0.788388,14.65,89.95,190.85,0.65,76.40,295.30,-3.764286,1.566667,3.238333
16800,314-89994,2025-12-30,1.40,0.00,-3.80,0.0498,False,0.350657,0.405797,0.757767,13.35,88.75,191.50,0.45,75.05,295.10,-3.642857,1.401667,3.129444


In [16]:
df.describe().transpose()

,count,mean,min,25%,50%,75%,max,std
date,16802,2002-12-31 12:00:00,1980-01-01 00:00:00,1991-07-02 06:00:00,2002-12-31 12:00:00,2014-07-01 18:00:00,2025-12-31 00:00:00,NaN
gts_runoff_daily,16802.0,1.125586,0.0,0.35,0.7,1.35,12.85,1.298625
gts_precip_daily,16802.0,2.440686,0.0,0.0,0.1,2.6,84.95,5.052907
gts_temp_daily,16802.0,4.482627,-27.65,-1.2,4.45,11.45,23.2,8.386351
gts_q_daily_m3ps,16802.0,0.040074,0.0,0.0125,0.0249,0.0481,0.4575,0.046242
gts_runoff_daily_log,16802.0,-0.285988,-3.912023,-0.994252,-0.328504,0.314811,2.554899,0.896767
gts_runoff_daily_rel_q95,16802.0,0.326257,0.0,0.101449,0.202899,0.391304,3.724638,0.376413
gts_runoff_daily_ecdf,16802.0,0.50003,0.000208,0.235151,0.498601,0.74625,1.0,0.288521
gts_runoff_7d_sum,16796.0,7.878873,0.1,2.7,5.05,9.65,79.6,8.591931
gts_runoff_30d_sum,16773.0,33.737706,1.25,13.45,24.1,43.7,211.25,30.163764


In [17]:
annual_df

,year,runoff_annual,precip_annual,temp_annual,site_code
0,1980,303.89,868.21,3.23,001-119928
1,1981,322.40,895.06,3.17,001-119928
2,1982,367.49,912.92,4.54,001-119928
3,1983,288.91,737.78,5.19,001-119928
4,1984,275.10,846.69,4.85,001-119928
...,...,...,...,...,...
390167,2021,441.45,531.90,4.67,314-89994
390168,2022,228.75,885.70,5.42,314-89994
390169,2023,618.15,903.50,4.61,314-89994
390170,2024,596.55,941.45,5.43,314-89994


In [18]:
annual_df.describe()

,year,runoff_annual,precip_annual,temp_annual
count,390172.000000,389988.000000,390126.000000,390126.00000
mean,2002.500000,1301.027546,1438.768166,3.72650
std,13.275935,1144.741786,754.247281,2.68803
min,1980.000000,24.100000,236.470000,-8.81000
25%,1991.000000,526.000000,918.200000,1.89000
50%,2002.500000,880.765000,1184.720000,4.14000
75%,2014.000000,1657.200000,1746.980000,5.77000
max,2025.000000,9775.600000,6295.830000,9.91000


In [19]:
len(annual_df.loc[annual_df['runoff_annual'] < 100]['site_code'].unique())

89

In [20]:
site_summary_df.describe()

,gts_runoff_q95,runoff_mean_annual,precip_mean_annual,temp_mean_annual
count,8478.000000,8478.000000,8481.000000,8481.000000
mean,11.957765,1301.027543,1438.768158,3.726500
std,8.409749,1091.795300,696.502301,2.486591
min,1.100000,108.226000,432.127000,-4.772000
25%,5.690500,549.676000,962.242000,1.961000
50%,9.438000,870.553500,1161.935000,4.281000
75%,15.975000,1665.692250,1751.226000,5.655000
max,49.600000,6686.449000,4387.908000,8.081000


In [21]:
site_summary_df.loc[site_summary_df['runoff_mean_annual'] < 150]

,site_code,gts_runoff_q95,has_runoff,has_precip,has_temp,runoff_mean_annual,precip_mean_annual,temp_mean_annual
287,002-118869,1.3,True,True,True,138.852,748.759,5.056
2269,012-39567,1.1,True,True,True,108.226,559.659,3.527
8252,212-83910,1.8,True,True,True,147.070,537.170,1.574
8283,224-96607,1.4,True,True,True,145.770,440.224,0.605
8366,246-91041,1.6,True,True,True,144.178,555.724,0.122


# Test reading daily data in again

In [22]:
# # Get list of available columns
# import pyarrow.parquet as pq

# schema = pq.read_schema(daily_out_dir / "gts_daily.parquet")
# schema.names

In [23]:
# daily_fpath = daily_out_dir / "gts_daily.parquet"

# cols = [
#     "site_code",
#     "date",
#     "gts_precip_mm_30d_sum",
#     "gts_temp_30d_mean",
#     "gts_runoff_mm"
# ]

# df = pd.read_parquet(daily_fpath, columns=cols)

# daily_df["date"] = pd.to_datetime(daily_df["date"])

# daily_df = daily_df.sort_values(
#     ["site_code", "date"]
# ).reset_index(drop=True)

# df